In [2]:
%pip install pretty_midi torch numpy matplotlib scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.6/5.6 MB 70.3 MB/s eta 0:00:0000:0100:01
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 54.6/54.6 kB 5.1 MB/s eta 0:00:00
  Created wheel for pretty_midi: filename=pretty_midi-0.2.11-py3-none-any.whl size=5595886 sha256=d2a23829f12f35acb30da3c9901a8dc1e2d940c5f302f890d12dd5eea71e99b5
  Stored in directory: /root/.cache/pip/wheels/f4/ad/93/a7042fe12668827574927ade9deec7f29aad2a1001b1501882
Successfully built pretty_midi


In [ ]:
from pathlib import Path
import pretty_midi
import numpy as np

DATA_ROOT = Path("/Users/catherinembata/music-generation-project/data/classic-midi-files-raw")
COMPOSERS = ["Bach", "Beethoven"]

midi_files = []
for comp in COMPOSERS:
    comp_dir = DATA_ROOT / comp
    midi_files.extend(sorted(comp_dir.rglob("*.mid")))

print(f"Found {len(midi_files)} MIDI files total from {COMPOSERS}.")
print("First few files:")
for p in midi_files[:10]:
    print(" ", p)
                      

Found 0 MIDI files total from ['Bach', 'Beethoven'].
First few files:


In [24]:
def choose_melody_instrument(midi: pretty_midi.PrettyMIDI):
    """Return a single pretty_midi.Instrument to treat as the melody, or None."""
    # Filter out drums
    candidates = [inst for inst in midi.instruments if not inst.is_drum]
    if not candidates:
        return None
    
    # Prefer Acoustic Grand Piano (program 0), but fall back to all non-drum
    piano_candidates = [inst for inst in candidates if inst.program == 0]
    if piano_candidates:
        candidates = piano_candidates
    
    # If instrument has no notes, skip it
    candidates = [inst for inst in candidates if inst.notes]
    if not candidates:
        return None
    
    # Pick instrument with highest average pitch (melody heuristic)
    def avg_pitch(inst):
        return np.mean([n.pitch for n in inst.notes])
    
    melody_inst = max(candidates, key=avg_pitch)
    return melody_inst


def duration_to_bucket(duration: float) -> int:
    """
    Map a duration in seconds to a small number of buckets.
    You can tweak these thresholds later.
    """
    if duration < 0.2:
        return 0  # very short (eighth-ish)
    elif duration < 0.5:
        return 1  # short (quarter-ish)
    elif duration < 1.0:
        return 2  # medium (half-ish)
    else:
        return 3  # long (whole+)


def notes_to_tokens(notes):
    """
    Convert a list of pretty_midi.Note objects into (pitch, duration_bucket) tokens,
    sorted by start time.
    """
    # Sort notes by start time
    notes = sorted(notes, key=lambda n: n.start)
    
    tokens = []
    for note in notes:
        pitch = note.pitch
        duration = note.end - note.start
        bucket = duration_to_bucket(duration)
        tokens.append((pitch, bucket))
    return tokens


In [25]:
# Build token sequences from ALL discovered MIDI files

SEQ_LEN = 50  # context window length

all_token_seqs = []      # list of lists of (pitch, bucket)
seq_file_paths = []      # parallel list of file paths (as strings)

num_loaded = 0
num_skipped_too_short = 0
num_skipped_no_melody = 0
num_failed_load = 0

max_verbose = 10  # only print detailed info for the first N files

for idx, midi_path in enumerate(midi_files):
    verbose = idx < max_verbose

    if verbose:
        print(f"Processing {midi_path}")

    # Try loading MIDI
    try:
        midi = pretty_midi.PrettyMIDI(str(midi_path))
        num_loaded += 1
    except Exception as e:
        if verbose:
            print(f"  Failed to load: {e}")
        num_failed_load += 1
        continue

    # Choose melody instrument
    melody = choose_melody_instrument(midi)
    if melody is None:
        if verbose:
            print("  No usable melody instrument, skipping.")
        num_skipped_no_melody += 1
        continue

    # Convert notes to tokens
    tokens = notes_to_tokens(melody.notes)
    if verbose:
        print(f"  Found {len(tokens)} tokens.")

    # Skip if not enough tokens to form at least one sequence
    if len(tokens) < SEQ_LEN + 1:
        if verbose:
            print("  Not enough tokens for sequence building, skipping.")
        num_skipped_too_short += 1
        continue

    all_token_seqs.append(tokens)
    seq_file_paths.append(str(midi_path))

# Summary
print("\n=== Summary ===")
print("Total files found:        ", len(midi_files))
print("Successfully loaded:      ", num_loaded)
print("Skipped (no melody):      ", num_skipped_no_melody)
print("Skipped (too short):      ", num_skipped_too_short)
print("Failed to load:           ", num_failed_load)
print("Usable token sequences:   ", len(all_token_seqs))


Processing /Users/catherinembata/music-generation-project/data/classic-midi-files-raw/Bach/AveMaria.mid
  Found 107 tokens.
Processing /Users/catherinembata/music-generation-project/data/classic-midi-files-raw/Bach/Bwv ''Little Notebook for Anna Magdalena Bach''/01 Menuet.mid
  Found 186 tokens.
Processing /Users/catherinembata/music-generation-project/data/classic-midi-files-raw/Bach/Bwv ''Little Notebook for Anna Magdalena Bach''/02 Menuet.mid
  Found 132 tokens.
Processing /Users/catherinembata/music-generation-project/data/classic-midi-files-raw/Bach/Bwv ''Little Notebook for Anna Magdalena Bach''/03 Menuet.mid
  Found 256 tokens.
Processing /Users/catherinembata/music-generation-project/data/classic-midi-files-raw/Bach/Bwv ''Little Notebook for Anna Magdalena Bach''/04 Menuet.mid
  Found 334 tokens.
Processing /Users/catherinembata/music-generation-project/data/classic-midi-files-raw/Bach/Bwv ''Little Notebook for Anna Magdalena Bach''/05 Polonaise.mid
  Found 592 tokens.
Processi

/Users/catherinembata/music-generation-project/.venv/lib/python3.9/site-packages/pretty_midi/pretty_midi.py:122: RuntimeWarning: Tempo, Key or Time signature change events found on non-zero tracks.  This is not a valid type 0 or type 1 MIDI file.  Tempo, Key or Time Signature may be wrong.
  warnings.warn(



=== Summary ===
Total files found:         1137
Successfully loaded:       1136
Skipped (no melody):       0
Skipped (too short):       157
Failed to load:            1
Usable token sequences:    979


In [26]:
# Build vocabulary from all (pitch, duration_bucket) tokens

# Flatten tokens from all sequences
all_tokens_flat = [tok for seq in all_token_seqs for tok in seq]

unique_tokens = sorted(set(all_tokens_flat))
token_to_id = {tok: idx for idx, tok in enumerate(unique_tokens)}
id_to_token = {idx: tok for tok, idx in token_to_id.items()}

print("Vocab size:", len(token_to_id))
print("First 20 vocab entries:")
for item in list(token_to_id.items())[:20]:
    print("  ", item)

# Optional: quick pitch range sanity check
pitches = [p for (p, b) in unique_tokens]
print("\nPitch range: min =", min(pitches), ", max =", max(pitches))


Vocab size: 307
First 20 vocab entries:
   ((24, 3), 0)
   ((26, 1), 1)
   ((26, 3), 2)
   ((27, 1), 3)
   ((28, 1), 4)
   ((28, 2), 5)
   ((28, 3), 6)
   ((29, 0), 7)
   ((29, 1), 8)
   ((29, 2), 9)
   ((29, 3), 10)
   ((30, 0), 11)
   ((30, 1), 12)
   ((30, 2), 13)
   ((30, 3), 14)
   ((31, 0), 15)
   ((31, 1), 16)
   ((31, 2), 17)
   ((31, 3), 18)
   ((32, 0), 19)

Pitch range: min = 24 , max = 107


In [27]:
# Build X (contexts), y (next-token targets), and example_file_indices

X_ids = []
y_ids = []
example_file_indices = []  # index into seq_file_paths

for file_idx, seq in enumerate(all_token_seqs):
    # Convert (pitch, bucket) tokens to IDs
    seq_ids = [token_to_id[t] for t in seq]

    if len(seq_ids) <= SEQ_LEN:
        continue  # already filtered earlier, but just in case

    # Slide a window of length SEQ_LEN
    for i in range(len(seq_ids) - SEQ_LEN):
        context = seq_ids[i : i + SEQ_LEN]
        target = seq_ids[i + SEQ_LEN]

        X_ids.append(context)
        y_ids.append(target)
        example_file_indices.append(file_idx)

X_ids = np.array(X_ids, dtype=np.int64)
y_ids = np.array(y_ids, dtype=np.int64)
example_file_indices = np.array(example_file_indices, dtype=np.int64)

print("X shape:", X_ids.shape)
print("y shape:", y_ids.shape)
print("Number of examples:", len(X_ids))
print("Unique source files used:", len(set(example_file_indices)))


X shape: (683118, 50)
y shape: (683118,)
Number of examples: 683118
Unique source files used: 979


In [28]:
from pathlib import Path
import numpy as np

# Where to save
out_dir = Path("/Users/catherinembata/music-generation-project/data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

out_path = out_dir / "full_sequences.npz"

# Save vocab in a convenient numpy form: (pitch, bucket, id)
vocab_array = np.array(
    [(p, b, token_to_id[(p, b)]) for (p, b) in unique_tokens],
    dtype=np.int64
)

np.savez(
    out_path,
    X=X_ids,
    y=y_ids,
    file_indices=example_file_indices,
    vocab=vocab_array,
)

print("Saved full dataset to:", out_path)
print("  X shape:", X_ids.shape)
print("  y shape:", y_ids.shape)
print("  vocab size:", len(unique_tokens))
print("  number of source files:", len(seq_file_paths))


Saved full dataset to: /Users/catherinembata/music-generation-project/data/processed/full_sequences.npz
  X shape: (683118, 50)
  y shape: (683118,)
  vocab size: 307
  number of source files: 979


In [29]:
import numpy as np
from sklearn.model_selection import train_test_split

# Unique file IDs (0 .. 978)
unique_files = np.unique(example_file_indices)
n_files = len(unique_files)
print("Total unique source files:", n_files)

# First split: train vs (val+test) – 70% / 30%
train_files, temp_files = train_test_split(
    unique_files,
    test_size=0.30,
    random_state=42,
    shuffle=True,
)

# Second split: val vs test – split temp 50/50 → 15% / 15% overall
val_files, test_files = train_test_split(
    temp_files,
    test_size=0.50,
    random_state=42,
    shuffle=True,
)

print("Train files:", len(train_files))
print("Val files:  ", len(val_files))
print("Test files: ", len(test_files))

# Build boolean masks over examples
train_mask = np.isin(example_file_indices, train_files)
val_mask   = np.isin(example_file_indices, val_files)
test_mask  = np.isin(example_file_indices, test_files)

# Convert masks to index arrays (positions in X_ids / y_ids)
train_idx = np.where(train_mask)[0]
val_idx   = np.where(val_mask)[0]
test_idx  = np.where(test_mask)[0]

print("\nExamples per split:")
print("  Train examples:", len(train_idx))
print("  Val examples:  ", len(val_idx))
print("  Test examples: ", len(test_idx))

# Sanity check: should cover all examples without overlap
total = len(train_idx) + len(val_idx) + len(test_idx)
print("\nTotal examples accounted for:", total, " (expected:", len(X_ids), ")")


Total unique source files: 979
Train files: 685
Val files:   147
Test files:  147

Examples per split:
  Train examples: 506185
  Val examples:   86984
  Test examples:  89949

Total examples accounted for: 683118  (expected: 683118 )


In [30]:
from pathlib import Path
import numpy as np

# Directory where you already saved full_sequences.npz
out_dir = Path("/Users/catherinembata/music-generation-project/data/processed")
out_dir.mkdir(parents=True, exist_ok=True)

splits_path = out_dir / "splits_filelevel_indices.npz"

np.savez(
    splits_path,
    train_idx=train_idx,
    val_idx=val_idx,
    test_idx=test_idx,
    train_files=train_files,
    val_files=val_files,
    test_files=test_files,
)

print("Saved split indices to:", splits_path)
print("  Train examples:", len(train_idx))
print("  Val examples:  ", len(val_idx))
print("  Test examples: ", len(test_idx))


Saved split indices to: /Users/catherinembata/music-generation-project/data/processed/splits_filelevel_indices.npz
  Train examples: 506185
  Val examples:   86984
  Test examples:  89949


In [31]:
import numpy as np

# Majority-token baseline: always predict the most frequent y in the TRAIN set

# 1. Get train/val/test targets
y_train = y_ids[train_idx]
y_val   = y_ids[val_idx]
y_test  = y_ids[test_idx]

# 2. Find the most common token in training labels
(unique, counts) = np.unique(y_train, return_counts=True)
majority_token_id = unique[np.argmax(counts)]
majority_count = counts.max()

print("Majority token ID:", majority_token_id)
print("Appears in train labels:", majority_count, "times")

# 3. Compute accuracy on each split
def majority_baseline_accuracy(y, majority_id):
    return np.mean(y == majority_id)

train_acc = majority_baseline_accuracy(y_train, majority_token_id)
val_acc   = majority_baseline_accuracy(y_val, majority_token_id)
test_acc  = majority_baseline_accuracy(y_test, majority_token_id)

print("\nMajority-token baseline accuracy:")
print(f"  Train: {train_acc:.4f}")
print(f"  Val:   {val_acc:.4f}")
print(f"  Test:  {test_acc:.4f}")
 

Majority token ID: 187
Appears in train labels: 18602 times

Majority-token baseline accuracy:
  Train: 0.0367
  Val:   0.0418
  Test:  0.0389
